<a href="https://colab.research.google.com/github/iamtrask/abcGPT/blob/main/notebooks/train_dual_karpathy_sts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# abcGPT dual-source — Karpathy-matched (Shakespeare + TinyStories char)

Per-slot architecture identical to nanoGPT's `config/train_shakespeare_char.py`:
- `n_layer=6, n_head=6, n_embd=384, dropout=0.2`
- `lr=1e-3, beta2=0.99, batch_size=64`, no grad accum
- `max_iters=10000` (Karpathy ran 5000 shake-batch iters; under alternating Uniform W_s steps every other pass, so 10000 total iters gives W_s the same 5000 effective updates)
- `batch_mode=alternating`, `mix_distribution=uniform` — Andrew's preferred recipe (alternating beat alt_mixed on wiki; Uniform beat Beta_half inside alt_mixed; alternating+Uniform inferred-best but this is its first full run)

Each slot at its native corner is doing the task Karpathy solved at val 1.4697. Whatever gap shows up between this run's shake-corner val and 1.4697 is the measurable cost of dual training. At every alpha the user samples from a ~10.65M-param effective model (the dual scheme stores two slot endpoints to define the slider's range; the 2x storage is not a capacity bump).

Compute estimate: ~40-50 min on T4.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install --quiet zstandard tiktoken

In [ ]:
import os
if not os.path.exists('/content/abcGPT'):
    !git clone --depth 1 https://github.com/iamtrask/abcGPT.git /content/abcGPT
else:
    !cd /content/abcGPT && git pull --rebase --autostash
%cd /content/abcGPT

In [ ]:
!python data/shakespeare_tinystories_char/prepare.py

## Run directory (tagged `-karpathy-sts` so this run is distinct from the others)

In [ ]:
RUN_ID = None   # set explicitly to resume, e.g. "20260519-XXXXXX-karpathy-sts"

In [ ]:
import time, os, glob
DRIVE_ROOT = '/content/drive/MyDrive/abcGPT/runs'
os.makedirs(DRIVE_ROOT, exist_ok=True)
if 'RUN_ID' not in dir() or RUN_ID is None:
    RUN_ID = time.strftime('%Y%m%d-%H%M%S') + '-karpathy-sts'
elif not RUN_ID.endswith('-karpathy-sts'):
    RUN_ID = RUN_ID + '-karpathy-sts'
OUT_DIR_DUAL = f'{DRIVE_ROOT}/{RUN_ID}'
os.makedirs(OUT_DIR_DUAL, exist_ok=True)
print('run dir:', OUT_DIR_DUAL)
snaps = sorted(glob.glob(os.path.join(OUT_DIR_DUAL, '*.pt.zst')))
print(f'  {len(snaps)} existing snapshots' + (' (will resume)' if snaps else ' (fresh run)'))

## Train

On a T4 expect roughly 40-50 minutes for 10000 iters. Pass time will be ~20-25s/pass (vs ~65s for the medium 38M-total model).

In [ ]:
!python train_dual.py config/train_shakespeare_tinystories_dual_karpathy.py \
    --out_dir=$OUT_DIR_DUAL \
    --batch_mode=alternating \
    --first_pass_corpus=shake \
    --mix_distribution=uniform